# Qwen3.5-27B INT8 (group-128)을 Colab에서 실행하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/infer-opt/blob/main/notebooks/qwen3_5_27b_int8_colab.ipynb)

[`Jakemu/Qwen3.5-27B-int8-g128`](https://huggingface.co/Jakemu/Qwen3.5-27B-int8-g128)에 올린 **weight-only INT8 Qwen3.5-27B**로 글을 생성하고 decode 속도를 잽니다.

- Linear 가중치(lm_head와 출력 32개짜리 delta-rule gate 제외)를 int8로 저장하고, 출력 채널마다·입력 128열마다 bf16 scale을 둡니다.
  가중치가 약 절반이 되어 batch 1 decode가 빨라집니다.
- 1 token decode의 행렬곱은 [infer-opt](https://github.com/Jake-Song/infer-opt)의 Triton 커널(`infer_opt/quant_kernels.py`)이 처리하고,
  decode step 전체는 `torch.compile` + CUDA graph로 돌립니다.
- A100 80GB에서 잰 값: bf16 35.2 ms/token → 이 모델 **20.7 ms/token**. HF 구현 대비 정확도 기준(같은 실행 bf16 KL의 2배 이하,
  확신 있는 불일치 없음)을 이 코드로 세 번 모두 통과했습니다.

**런타임:** 런타임 → 런타임 유형 변경 → **A100 GPU**. 커널이 bf16을 쓰므로 compute capability 8.0 이상이 필요합니다.
GPU 메모리는 약 30 GiB라 A100 40GB에도 들어갑니다(원래 bf16 모델은 가중치만 52 GiB). 다운로드는 약 30 GB입니다.

이 checkpoint는 `transformers` 형식이 아니라서 infer-opt 코드로 읽습니다. 셀을 위에서부터 차례로 실행하세요.

## 1. GPU 확인

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다: 런타임 → 런타임 유형 변경 → A100 GPU"
name, cap = torch.cuda.get_device_name(), torch.cuda.get_device_capability()
print(f"{name}, compute capability {cap[0]}.{cap[1]}, torch {torch.__version__}")
assert cap >= (8, 0), "bf16 커널은 compute capability 8.0 이상(A100, H100 등)이 필요합니다"


## 2. 라이브러리와 코드 설치

- `transformers`: tokenizer와 chat template를 씁니다.
- `flash-linear-attention`: delta-rule 커널(`fla-delta`).
- infer-opt는 GitHub에서 받아 커밋 `661f694a9e76`로 고정합니다(이 노트북을 만들 때의 코드).

In [ ]:
!pip install -q -U "transformers>=5" flash-linear-attention
!rm -rf /content/infer-opt && git clone -q https://github.com/Jake-Song/infer-opt.git /content/infer-opt
!git -C /content/infer-opt checkout -q 661f694a9e767e3c26910d8c13336bcc7f1c222a && git -C /content/infer-opt log -1 --format="%h %s"

import sys
sys.path.insert(0, "/content/infer-opt")


## 3. 모델 받기와 불러오기

Hub에서 int8 checkpoint(safetensors shard와 `quantization.json`)를 받고, `from_quantized_dir`로 GPU에 바로 올립니다.
`quantization.json`에는 어떤 Linear가 int8인지와 group 크기가 들어 있습니다.
`fuse_projections()`는 같은 입력을 읽는 투영(gate+up, q+k+v, in_proj_qkv+in_proj_z)의 int8 행렬과 scale을 행으로 쌓아
decode step의 행렬곱을 하나씩으로 줄입니다. 읽는 바이트는 같고 kernel 실행이 줄어 빨라집니다(checkpoint는 그대로).

In [ ]:
import time
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from infer_opt.quant import Int8Linear
from infer_opt.qwen3_5 import Qwen3_5ForCausalLM

MODEL_ID = "Jakemu/Qwen3.5-27B-int8-g128"
t0 = time.perf_counter()
path = Path(snapshot_download(MODEL_ID))
size = sum(f.stat().st_size for f in path.glob("*.safetensors"))
print(f"다운로드 {size / 2**30:.2f} GiB, {time.perf_counter() - t0:.0f} s")

t0 = time.perf_counter()
DTYPE = torch.bfloat16
model = Qwen3_5ForCausalLM.from_quantized_dir(path, device="cuda", kernels="fla-delta", dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(path)
print("합친 투영 묶음", model.fuse_projections())
n_int8 = sum(isinstance(m, Int8Linear) for m in model.modules())
print(f"불러오기 {time.perf_counter() - t0:.0f} s, Int8Linear {n_int8}개, GPU 메모리 {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


## 4. Decode 준비

- **static cache**: K/V와 delta-rule 상태를 최대 길이(`MAX_LEN`)만큼 미리 잡아 둡니다. 프롬프트가 바뀌면 `reset()`만 합니다.
  고정 shape 경로의 full attention은 cache 길이 전체를 훑으므로 `MAX_LEN`이 길수록 step이 조금 느려집니다.
- **`FixedShapeDecoder(..., "compile-reduce-overhead")`**: 1 token decode step을 `torch.compile` + CUDA graph로 돌립니다.
  처음 step에서 컴파일하므로 **첫 생성은 2–4분** 걸립니다(prefill의 Triton·FLA 커널도 처음 쓸 때 컴파일되어 첫 prefill만 수십 초).
  그 뒤로는 같은 그래프를 재생합니다.
- inductor의 `max_autotune`과 `coordinate_descent_tuning`은 벤치마크를 잴 때와 같은 설정입니다.
- prompt(prefill)는 컴파일 없이 한 번에 처리합니다. 이 경로는 int8 가중치를 `DTYPE`으로 풀어서 곱하므로 decode보다 메모리를 더 씁니다.

In [ ]:
import torch._inductor.config as inductor_config

from infer_opt.qwen3_5 import FixedShapeDecoder

inductor_config.max_autotune = True
inductor_config.coordinate_descent_tuning = True

MAX_LEN = 2048
cache = model.new_cache(1, MAX_LEN)
decoder = FixedShapeDecoder(model, cache, "compile-reduce-overhead")
stop_ids = {tokenizer.eos_token_id, *tokenizer.convert_tokens_to_ids(["<|im_end|>", "<|endoftext|>"])} - {None}


@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 256, stream: bool = True) -> dict:
    """chat template로 감싼 prompt에 greedy decoding. 생성 글과 prefill / decode 시간을 돌려준다."""
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.cuda()
    if ids.shape[1] + max_new_tokens > MAX_LEN:
        raise ValueError(f"prompt {ids.shape[1]} + {max_new_tokens} tokens > MAX_LEN {MAX_LEN}")
    cache.reset()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    token = model(ids, cache)[:, -1].argmax(-1, keepdim=True)  # prefill
    torch.cuda.synchronize(); t1 = time.perf_counter()
    out, shown = [int(token)], ""
    for _ in range(max_new_tokens - 1):
        if out[-1] in stop_ids:
            break
        token = decoder.step(token)[:, -1].argmax(-1, keepdim=True)
        out.append(int(token))  # int()이 GPU를 기다리므로 step마다 동기화된다
        if stream:
            now = tokenizer.decode(out, skip_special_tokens=True)
            if not now.endswith("�"):  # 여러 token에 걸친 글자는 다 모일 때까지 기다린다
                print(now[len(shown):], end="", flush=True)
                shown = now
    torch.cuda.synchronize(); t2 = time.perf_counter()
    steps = len(out) - 1
    stats = {"prompt_tokens": ids.shape[1], "new_tokens": len(out), "prefill_s": t1 - t0,
             "decode_ms_per_token": (t2 - t1) / max(steps, 1) * 1e3}
    if stream:
        print(f"\n\n[prompt {stats['prompt_tokens']} tok, 생성 {stats['new_tokens']} tok, prefill {stats['prefill_s']:.2f} s, "
              f"decode {stats['decode_ms_per_token']:.1f} ms/token]")
    return {"text": tokenizer.decode(out, skip_special_tokens=True), "tokens": out, **stats}


## 5. 생성해 보기

첫 실행은 컴파일 때문에 오래 걸리고, 표시되는 decode ms/token에도 컴파일 시간이 섞입니다. 두 번째부터가 실제 속도입니다.
글자를 하나씩 출력하느라 step마다 GPU를 기다리므로, 여기 나오는 ms/token은 6절의 측정보다 조금 큽니다.

In [ ]:
_ = generate("Explain in three sentences why batch-1 LLM decoding is limited by memory bandwidth.", max_new_tokens=160)

In [ ]:
_ = generate("가중치를 int8로 양자화하면 언어 모델 추론이 왜 빨라지는지, 정확도는 어떻게 확인하는지 다섯 문장으로 설명해 줘.",
             max_new_tokens=256)

## 6. Decode 속도 측정

벤치마크 스크립트(`scripts/bench_qwen3_5_int8_decode.py`)와 같은 방식입니다: 랜덤 1024-token 프롬프트를 prefill하고,
warm-up 3 step 뒤 128 step을 이어서 돌린 평균 ms/token입니다. 출력도 동기화도 step 사이에 하지 않습니다.
cache 길이도 그 스크립트와 같게(1211) 따로 잡으므로 decode step을 한 번 더 컴파일합니다(약 2분).
같은 방식으로 A100-SXM4-80GB에서 잰 값은 bf16 35.17, 이 모델 20.72 ms/token입니다(투영을 합치기 전의 코드로는 25.93).
A100-SXM4-40GB(메모리 대역폭이 약 3/4)에서 이 노트북으로 잰 값은 24.13 ms/token, peak 29.1 GiB였습니다.

In [ ]:
@torch.no_grad()
def bench_decode(prompt_tokens: int = 1024, steps: int = 128, warmup: int = 3) -> float:
    bench_cache = model.new_cache(1, prompt_tokens + warmup + steps + 56)  # 1211: 벤치마크 스크립트와 같은 cache 길이
    bench_decoder = FixedShapeDecoder(model, bench_cache, "compile-reduce-overhead")
    torch.manual_seed(0)
    ids = torch.randint(1000, 100000, (1, prompt_tokens), device="cuda")
    token = model(ids, bench_cache)[:, -1].argmax(-1, keepdim=True)
    for _ in range(warmup):  # 첫 step에서 컴파일된다
        token = bench_decoder.step(token)[:, -1].argmax(-1, keepdim=True)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(steps):
        token = bench_decoder.step(token)[:, -1].argmax(-1, keepdim=True)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / steps * 1e3


ms = bench_decode()
print(f"{torch.cuda.get_device_name()}: {ms:.2f} ms/token ({1000 / ms:.1f} tok/s), "
      f"peak GPU 메모리 {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")


## 7. HF 모델과의 비교는?

9B용 A100 노트북에는 원래 모델(`transformers`, bf16)과의 비교가 있지만 27B는 할 수 없습니다. 원래 모델은 가중치만 52 GiB라
이 INT8 모델(약 28 GiB)과 함께 80 GB에도 들어가지 않습니다. HF bf16과 비교한 정확도 기준의 결과는 저장소의
`worklog.md`(2026-10-20, Qwen3.5-27B와 그 최적화)에 적어 두었습니다: 이 노트북의 코드로 세 번 실행(Hub에서 받은 이 checkpoint 0.86배,
제자리 양자화 1.07배, lm_head까지 int8 0.67배)에서 HF 대비 KL이 같은 실행 bf16 구현의 0.67–1.07배였고,
확신 있는 불일치(margin > 0.5)는 없었습니다.

## 8. 직접 써 보기

`PROMPT`를 바꿔서 실행하세요. greedy decoding(항상 가장 확률 높은 token)만 지원합니다.

In [ ]:
PROMPT = "Qwen3.5의 linear attention(gated delta rule)이 긴 문맥에서 유리한 이유를 설명해 줘."
_ = generate(PROMPT, max_new_tokens=300)

## 참고

- 측정과 정확도 기준의 원시 결과는 저장소의 `notebooks/assets/qwen3_5_27b_int8_opt_a100_20261020/`(이 코드)와
  `notebooks/assets/qwen3_5_27b_int8_a100_20261020/`(처음 양자화했을 때), `worklog.md`에 있습니다.
- (A100 80GB 기준) lm_head까지 int8로 하면 20.31 ms/token으로 조금 더 빠르고 정확도 기준도 통과했지만, 9B와 같게 lm_head는 bf16으로 둡니다.